In [1]:
import warnings
warnings.filterwarnings("ignore")

In [2]:
import yfinance as yf
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import time

def get_minute_bars(ticker, time_period='1m'):
    """
    Fetch minute-by-minute price data for a given stock ticker as far back as yahoo finance allows.

    Args:
    ticker (str): The stock ticker symbol.

    Returns:
    pandas.DataFrame: A DataFrame containing the minute-by-minute price data.
    """

    # Initialize the ticker object
    stock = yf.Ticker(ticker)

    # Set the end date to now and initialize an empty DataFrame for all data
    end_date = datetime.now()
    all_data = pd.DataFrame()

    # Function to fetch data with error handling
    def fetch_data(ticker, start, end, interval=time_period):
        try:
            df = ticker.history(start=start, end=end, interval=interval)
            if df.empty:
                raise ValueError("No data returned")
            return df
        except Exception as e:
            print(f"Error fetching data: {str(e)}")
            return pd.DataFrame()

    # Fetch data in 7-day intervals
    while True:
        start_date = end_date - timedelta(days=7)
        data = fetch_data(stock, start_date, end_date)
        
        if data.empty:
            break
        
        all_data = pd.concat([data, all_data])
        end_date = start_date
        
        print(f"Fetched data from {start_date} to {end_date}")
        
        # Add a 1 second sleep between requests
        time.sleep(0.5)

    if not all_data.empty:
        print(f"\nData retrieved for {ticker}")
        print(f"Total number of minutes of data: {len(all_data)}")
        print(f"Date range: {all_data.index[-1]} to {all_data.index[0]}")
        return all_data
    else:
        print(f"No data available for {ticker}.")
        return None


In [3]:
all_data = get_minute_bars('MGC=F', time_period='5m')

# # Crypto
# all_data = get_minute_bars('BTC-USD', time_period='5m')

Fetched data from 2024-10-12 21:22:30.135046 to 2024-10-12 21:22:30.135046
Fetched data from 2024-10-05 21:22:30.135046 to 2024-10-05 21:22:30.135046
Fetched data from 2024-09-28 21:22:30.135046 to 2024-09-28 21:22:30.135046
Fetched data from 2024-09-21 21:22:30.135046 to 2024-09-21 21:22:30.135046
Fetched data from 2024-09-14 21:22:30.135046 to 2024-09-14 21:22:30.135046
Fetched data from 2024-09-07 21:22:30.135046 to 2024-09-07 21:22:30.135046
Fetched data from 2024-08-31 21:22:30.135046 to 2024-08-31 21:22:30.135046
Fetched data from 2024-08-24 21:22:30.135046 to 2024-08-24 21:22:30.135046


$BTC-USD: possibly delisted; no price data found  (5m 2024-08-17 21:22:30.135046 -> 2024-08-24 21:22:30.135046) (Yahoo error = "5m data not available for startTime=1723929750 and endTime=1724534550. The requested range must be within the last 60 days.")


Error fetching data: No data returned

Data retrieved for BTC-USD
Total number of minutes of data: 16121
Date range: 2024-10-19 20:55:00+00:00 to 2024-08-24 21:20:00+00:00


In [4]:
all_data

,Open,High,Low,Close,Volume,Dividends,Stock Splits
Datetime,,,,,,,
2024-08-24 21:20:00+00:00,64184.835938,64184.835938,64124.210938,64124.210938,0,0.0,0.0
2024-08-24 21:25:00+00:00,64083.449219,64083.449219,63971.738281,63975.136719,0,0.0,0.0
2024-08-24 21:30:00+00:00,63977.746094,64073.472656,63977.746094,64073.472656,0,0.0,0.0
2024-08-24 21:35:00+00:00,64071.328125,64071.328125,63857.519531,63874.375000,0,0.0,0.0
2024-08-24 21:40:00+00:00,63867.839844,63942.101562,63867.839844,63942.101562,0,0.0,0.0
...,...,...,...,...,...,...,...
2024-10-19 20:35:00+00:00,68218.187500,68242.742188,68212.851562,68242.742188,0,0.0,0.0
2024-10-19 20:40:00+00:00,68234.773438,68249.625000,68234.773438,68244.179688,86324224,0.0,0.0
2024-10-19 20:45:00+00:00,68247.195312,68247.195312,68232.031250,68236.796875,9773056,0.0,0.0


In [5]:
def mark_asia_open_zones(df, timeframe=4):
    """
    Mark the zones between Asia open and "timeframe" hours later on the dataframe.
    
    Args:
    df (pandas.DataFrame): The dataframe containing minute-by-minute price data.
    timeframe (int): The number of hours after Asia open to mark the zone.
    
    Returns:
    pandas.DataFrame: The dataframe with an additional 'Asia_Open_Zone' column.
    """
    # Ensure the index is timezone-aware in US/Eastern time
    df = df.tz_convert('US/Eastern')
    
    # Determine if each timestamp is in DST
    is_dst = df.index.map(lambda ts: ts.dst() != timedelta(0))

    # Map DST to asia_open_hour
    asia_open_hours = is_dst.map({True: 21, False: 20})

    # Compute asia_open_time and asia_close_time
    dates = df.index.normalize()
    asia_open_times = dates + pd.to_timedelta(asia_open_hours, unit='h')
    asia_close_times = asia_open_times + pd.Timedelta(hours=timeframe)

    # Create 'Asia_Open_Zone' column
    df['Asia_Open_Zone'] = (df.index >= asia_open_times) & (df.index < asia_close_times)
    
    return df

In [6]:
# Apply the function to the all_data dataframe
all_data = mark_asia_open_zones(all_data)

# Display the first few rows of the updated dataframe
print(all_data.head())

# Display some statistics about the Asia Open Zones
asia_open_data = all_data[all_data['Asia_Open_Zone']]
print(f"\nTotal number of minutes in Asia Open Zones: {len(asia_open_data)}")
print(f"Percentage of data in Asia Open Zones: {len(asia_open_data) / len(all_data) * 100:.2f}%")

                                   Open          High           Low  \
Datetime                                                              
2024-08-24 17:20:00-04:00  64184.835938  64184.835938  64124.210938   
2024-08-24 17:25:00-04:00  64083.449219  64083.449219  63971.738281   
2024-08-24 17:30:00-04:00  63977.746094  64073.472656  63977.746094   
2024-08-24 17:35:00-04:00  64071.328125  64071.328125  63857.519531   
2024-08-24 17:40:00-04:00  63867.839844  63942.101562  63867.839844   

                                  Close  Volume  Dividends  Stock Splits  \
Datetime                                                                   
2024-08-24 17:20:00-04:00  64124.210938       0        0.0           0.0   
2024-08-24 17:25:00-04:00  63975.136719       0        0.0           0.0   
2024-08-24 17:30:00-04:00  64073.472656       0        0.0           0.0   
2024-08-24 17:35:00-04:00  63874.375000       0        0.0           0.0   
2024-08-24 17:40:00-04:00  63942.101562       

In [7]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go

def simulate_trading_strategy_stock(df, breakout_pct=0.2, starting_capital=10000, leverage=1, take_profit_pct=0.5, stop_loss_pct=0.5, enter_opposite_side=False):
    """
    Simulate the trading strategy with capital and leverage considerations.

    Args:
    df (pandas.DataFrame): The dataframe with minute-by-minute price data and 'Asia_Open_Zone' column.
    breakout_pct (float): Percentage of the range to consider as a breakout.
    trailing_stop_pct (float): Percentage of the range to set as the trailing stop.
    starting_capital (float): The starting capital for the trading simulation.
    leverage (float): The leverage to be used for each trade.

    Returns:
    pandas.DataFrame: A dataframe containing the details of each trade executed.
    """
    # Ensure the dataframe is sorted by index
    df = df.sort_index()

    # Create a unique ID for each Asia open session
    df['Asia_Open_Session_ID'] = (df['Asia_Open_Zone'] & (~df['Asia_Open_Zone'].shift(1).fillna(False))).cumsum()

    # Compute the high and low for each Asia open session
    asia_open_ranges = df[df['Asia_Open_Zone']].groupby('Asia_Open_Session_ID').agg({
        'High': 'max',
        'Low': 'min'
    }).rename(columns={'High': 'Asia_Open_High', 'Low': 'Asia_Open_Low'})

    # Forward-fill the session IDs and merge the Asia open ranges back into the dataframe
    df['Asia_Open_Session_ID'] = df['Asia_Open_Session_ID'].ffill()
    df = df.merge(asia_open_ranges, on='Asia_Open_Session_ID', how='left')

    # Calculate the range and breakout thresholds
    df['Asia_Open_Range'] = df['Asia_Open_High'] - df['Asia_Open_Low']
    df['Breakout_Threshold'] = breakout_pct * df['Asia_Open_Range']

    # Determine if each timestamp is after the Asia open session
    df['After_Asia_Open_Session'] = ~df['Asia_Open_Zone']

    # Initialize variables
    all_trades = []
    capital = starting_capital

    # Process each Asia open session individually
    for session_id, group in df.groupby('Asia_Open_Session_ID'):
        if capital <= 0:
            print("Capital exhausted. Stopping simulation.")
            break
        trades = process_trading_strategy_stock(group, breakout_pct, capital, leverage, take_profit_pct, stop_loss_pct, enter_opposite_side)

        for trade in trades:
            # Update capital based on the profit from the trade
            capital += trade['Profit']
            # Store capital after the trade
            trade['Capital_After_Trade'] = capital

        all_trades.extend(trades)

    # Convert the list of trades into a DataFrame
    trades_df = pd.DataFrame(all_trades)
    return trades_df, df

def process_trading_strategy_stock(
    group,
    breakout_pct,
    capital,
    leverage,
    take_profit_pct,
    stop_loss_pct,
    enter_opposite_side=False
):
    """
    Process the trading strategy for a single Asia open session with capital and leverage considerations.

    Args:
    group (pandas.DataFrame): The dataframe for a single Asia open session.
    breakout_pct (float): Percentage of the range to consider as a breakout.
    capital (float): The capital available before the trade.
    leverage (float): The leverage to be used for the trade.
    take_profit_pct (float): Percentage of the range to set as the take profit target.
    stop_loss_pct (float): Percentage of the range to set as the stop loss level.
    enter_opposite_side (bool): If True, enter at the opposite side of the range after first breakout.

    Returns:
    list: A list of dictionaries containing trade details.
    """
    # Only process data after the Asia open session
    group = group[group['After_Asia_Open_Session']]

    # Initialize variables
    in_breakout = False
    breakout_direction = None  # 'up' or 'down'
    reentered_range = False
    trade_executed = False
    trades = []

    # Get the Asia open high, low, and range
    if group.empty:
        return trades  # No data to process

    asia_open_high = group['Asia_Open_High'].iloc[0]
    asia_open_low = group['Asia_Open_Low'].iloc[0]
    asia_open_range = asia_open_high - asia_open_low
    breakout_threshold = breakout_pct * asia_open_range

    # Define breakout levels
    breakout_up_level = asia_open_high + breakout_threshold
    breakout_down_level = asia_open_low - breakout_threshold

    for idx, row in group.iterrows():
        if not in_breakout:
            # Check for initial breakout
            if row['High'] >= breakout_up_level:
                in_breakout = True
                breakout_direction = 'up'
                breakout_price = row['Close']
            elif row['Low'] <= breakout_down_level:
                in_breakout = True
                breakout_direction = 'down'
                breakout_price = row['Close']
        elif not reentered_range:
            # Check if price re-enters the range
            if (breakout_direction == 'up' and row['Low'] <= asia_open_low):
                reentered_range = True
            elif (breakout_direction == 'down' and row['High'] >= asia_open_high):
                reentered_range = True
        elif not trade_executed:
            if enter_opposite_side:
                # Enter at the opposite side of the range after re-entry
                if breakout_direction == 'up' and row['Low'] <= asia_open_low:
                    # Execute a long trade at the bottom of the range
                    trade_executed = True
                    entry_price = asia_open_low
                    trade_time = idx
                    stop_loss = entry_price - stop_loss_pct * asia_open_range
                    take_profit = entry_price + take_profit_pct * asia_open_range
                    trades.append({
                        'Time': trade_time,
                        'Direction': 'Long',
                        'Entry_Price': entry_price,
                        'Stop_Loss': stop_loss,
                        'Take_Profit': take_profit,
                        'Exit_Price': np.nan,
                        'Exit_Time': np.nan,
                        'Profit': np.nan,
                        'Capital_Before_Trade': capital,
                        'Leverage': leverage,
                        'Asia_Open_Session_ID': group['Asia_Open_Session_ID'].iloc[0]
                    })
                elif breakout_direction == 'down' and row['High'] >= asia_open_high:
                    # Execute a short trade at the top of the range
                    trade_executed = True
                    entry_price = asia_open_high
                    trade_time = idx
                    stop_loss = entry_price + stop_loss_pct * asia_open_range
                    take_profit = entry_price - take_profit_pct * asia_open_range
                    trades.append({
                        'Time': trade_time,
                        'Direction': 'Short',
                        'Entry_Price': entry_price,
                        'Stop_Loss': stop_loss,
                        'Take_Profit': take_profit,
                        'Exit_Price': np.nan,
                        'Exit_Time': np.nan,
                        'Profit': np.nan,
                        'Capital_Before_Trade': capital,
                        'Leverage': leverage,
                        'Asia_Open_Session_ID': group['Asia_Open_Session_ID'].iloc[0]
                    })
            else:
                # Original logic: Check for second breakout in the same direction
                if breakout_direction == 'up' and row['Low'] <= asia_open_low:
                    # Execute a long trade
                    trade_executed = True
                    entry_price = asia_open_low
                    trade_time = idx
                    stop_loss = entry_price - stop_loss_pct * asia_open_range
                    take_profit = asia_open_high + take_profit_pct * asia_open_range
                    trades.append({
                        'Time': trade_time,
                        'Direction': 'Long',
                        'Entry_Price': entry_price,
                        'Stop_Loss': stop_loss,
                        'Take_Profit': take_profit,
                        'Exit_Price': np.nan,
                        'Exit_Time': np.nan,
                        'Profit': np.nan,
                        'Capital_Before_Trade': capital,
                        'Leverage': leverage,
                        'Asia_Open_Session_ID': group['Asia_Open_Session_ID'].iloc[0]
                    })
                elif breakout_direction == 'down' and row['High'] >= asia_open_high:
                    # Execute a short trade
                    trade_executed = True
                    entry_price = asia_open_high
                    trade_time = idx
                    stop_loss = entry_price + stop_loss_pct * asia_open_range
                    take_profit = asia_open_low - take_profit_pct * asia_open_range
                    trades.append({
                        'Time': trade_time,
                        'Direction': 'Short',
                        'Entry_Price': entry_price,
                        'Stop_Loss': stop_loss,
                        'Take_Profit': take_profit,
                        'Exit_Price': np.nan,
                        'Exit_Time': np.nan,
                        'Profit': np.nan,
                        'Capital_Before_Trade': capital,
                        'Leverage': leverage,
                        'Asia_Open_Session_ID': group['Asia_Open_Session_ID'].iloc[0]
                    })
        elif trade_executed:
            # Manage the trade by checking stop loss and take profit
            current_trade = trades[-1]
            if current_trade['Direction'] == 'Long':
                # Check if take profit is hit
                if row['High'] >= current_trade['Take_Profit']:
                    exit_price = current_trade['Take_Profit']
                    exit_time = idx
                    profit = capital * leverage * ((exit_price - current_trade['Entry_Price']) / current_trade['Entry_Price'])
                    current_trade.update({
                        'Exit_Price': exit_price,
                        'Exit_Time': exit_time,
                        'Profit': profit
                    })
                    break  # Exit after trade is closed
                # Check if stop loss is hit
                elif row['Low'] <= current_trade['Stop_Loss']:
                    exit_price = current_trade['Stop_Loss']
                    exit_time = idx
                    profit = capital * leverage * ((exit_price - current_trade['Entry_Price']) / current_trade['Entry_Price'])
                    current_trade.update({
                        'Exit_Price': exit_price,
                        'Exit_Time': exit_time,
                        'Profit': profit
                    })
                    break  # Exit after trade is closed
            elif current_trade['Direction'] == 'Short':
                # Check if take profit is hit
                if row['Low'] <= current_trade['Take_Profit']:
                    exit_price = current_trade['Take_Profit']
                    exit_time = idx
                    profit = capital * leverage * ((current_trade['Entry_Price'] - exit_price) / current_trade['Entry_Price'])
                    current_trade.update({
                        'Exit_Price': exit_price,
                        'Exit_Time': exit_time,
                        'Profit': profit
                    })
                    break  # Exit after trade is closed
                # Check if stop loss is hit
                elif row['High'] >= current_trade['Stop_Loss']:
                    exit_price = current_trade['Stop_Loss']
                    exit_time = idx
                    profit = capital * leverage * ((current_trade['Entry_Price'] - exit_price) / current_trade['Entry_Price'])
                    current_trade.update({
                        'Exit_Price': exit_price,
                        'Exit_Time': exit_time,
                        'Profit': profit
                    })
                    break  # Exit after trade is closed

    # If trade is still open at the end of the day, exit at the close price
    if trade_executed and np.isnan(trades[-1]['Exit_Price']):
        current_trade = trades[-1]
        exit_price = group['Close'].iloc[-1]
        exit_time = group.index[-1]
        if current_trade['Direction'] == 'Long':
            profit = capital * leverage * ((exit_price - current_trade['Entry_Price']) / current_trade['Entry_Price'])
        elif current_trade['Direction'] == 'Short':
            profit = capital * leverage * ((current_trade['Entry_Price'] - exit_price) / current_trade['Entry_Price'])
        current_trade.update({
            'Exit_Price': exit_price,
            'Exit_Time': exit_time,
            'Profit': profit
        })

    return trades


def plot_trading_chart(df, trades_df):
    """
    Plot the OHLC data on a candlestick chart using Plotly, show the opening ranges,
    and display entries and exits on the chart.

    Args:
    df (pandas.DataFrame): The original dataframe with price data and Asia open ranges.
    trades_df (pandas.DataFrame): The dataframe containing the details of each trade executed.
    """
    # Create a Plotly figure
    fig = go.Figure()

    # Add candlestick trace
    fig.add_trace(go.Candlestick(
        x=df.index,
        open=df['Open'],
        high=df['High'],
        low=df['Low'],
        close=df['Close'],
        name='Price'
    ))

    # Add Asia opening ranges as horizontal boxes
    print("df:", df)
    asia_open_sessions = df[['Asia_Open_Session_ID', 'Asia_Open_High', 'Asia_Open_Low']].drop_duplicates()
    for _, session in asia_open_sessions.iterrows():
        session_id = session['Asia_Open_Session_ID']
        session_data = df[df['Asia_Open_Session_ID'] == session_id]
        # Get the time range for the session
        session_times = session_data.index
        if len(session_times) == 0:
            continue
        start_time = session_times[0]
        end_time = session_times[-1]
        asia_open_high = session['Asia_Open_High']
        asia_open_low = session['Asia_Open_Low']

        fig.add_shape(
            type="rect",
            x0=start_time,
            y0=asia_open_low,
            x1=end_time,
            y1=asia_open_high,
            line=dict(color="LightSeaGreen", width=1),
            fillcolor="LightSeaGreen",
            opacity=0.2,
            layer="below"
        )

    # Add trade entries and exits
    for idx, trade in trades_df.iterrows():
        entry_time = trade['Time']
        entry_price = trade['Entry_Price']
        exit_time = trade['Exit_Time']
        exit_price = trade['Exit_Price']
        direction = trade['Direction']

        # Entry marker
        fig.add_trace(go.Scatter(
            x=[entry_time],
            y=[entry_price],
            mode='markers',
            marker=dict(
                color='green' if direction == 'Long' else 'red',
                symbol='triangle-up' if direction == 'Long' else 'triangle-down',
                size=10
            ),
            name='Trade Entry'
        ))

        # Exit marker
        if pd.notna(exit_time):
            fig.add_trace(go.Scatter(
                x=[exit_time],
                y=[exit_price],
                mode='markers',
                marker=dict(
                    color='darkgreen' if direction == 'Long' else 'darkred',
                    symbol='x',
                    size=10
                ),
                name='Trade Exit'
            ))

            # Draw line from entry to exit
            fig.add_trace(go.Scatter(
                x=[entry_time, exit_time],
                y=[entry_price, exit_price],
                mode='lines',
                line=dict(
                    color='green' if direction == 'Long' else 'red',
                    dash='dash'
                ),
                showlegend=False
            ))

    # Update layout
    fig.update_layout(
        title='Trading Strategy Simulation',
        yaxis_title='Price',
        xaxis_title='Time',
        xaxis_rangeslider_visible=False,
        template='plotly_dark'
    )

    fig.show()

# Example usage:
if __name__ == "__main__":

    # Simulate the trading strategy with starting capital and leverage
    trades_df, df = simulate_trading_strategy_stock(all_data, breakout_pct=0.75, starting_capital=10000, leverage=20, take_profit_pct=2.5, stop_loss_pct=1, enter_opposite_side=True)

    # Calculate total profit
    total_profit = trades_df['Profit'].sum()
    print(f"Total Profit: {total_profit}")

    # Calculate win rate
    win_rate = (trades_df['Profit'] > 0).mean() * 100
    print(f"Win Rate: {win_rate:.2f}%")

    # Display the trades
    print(trades_df)

    # Plot the trading chart
    plot_trading_chart(df, trades_df)


Total Profit: 2812.241537744339
Win Rate: 41.67%
     Time Direction   Entry_Price     Stop_Loss   Take_Profit    Exit_Price  \
0    1079      Long  58966.785156  58298.792969  60636.765625  58298.792969   
1    1650      Long  58924.578125  58527.042969  59918.416016  58527.042969   
2    3645     Short  56821.250000  57652.398438  54743.378906  54743.378906   
3    4315     Short  54487.664062  54839.406250  53608.308594  54839.406250   
4    5138     Short  57736.582031  58761.488281  55174.316406  57637.554688   
5    7164     Short  60716.769531  61385.625000  59044.630859  61385.625000   
6    8337     Short  63285.742188  63718.027344  62205.029297  63718.027344   
7   10347     Short  65934.242188  66182.078125  65314.652344  65314.652344   
8   10849      Long  63171.828125  62664.593750  64439.914062  62664.593750   
9   12585     Short  63927.226562  64499.976562  62495.351562  62495.351562   
10  12876      Long  62363.933594  61943.367188  63415.349609  61943.367188   
11 

In [8]:
def simulate_trading_strategy_futures(
    df,
    breakout_pct=0.2,
    starting_capital=10000,
    margin_per_contract=400,
    contract_size=10,
    take_profit_pct=0.5,
    stop_loss_pct=0.5,
    enter_opposite_side=False,
):
    """
    Simulate the trading strategy with capital and futures contract considerations.

    Args:
    df (pandas.DataFrame): The dataframe with minute-by-minute price data and 'Asia_Open_Zone' column.
    breakout_pct (float): Percentage of the range to consider as a breakout.
    starting_capital (float): The starting capital for the trading simulation.
    margin_per_contract (float): The margin required per contract.
    contract_size (int): The contract size (e.g., 10 troy ounces for MGC).
    take_profit_pct (float): Percentage of the range to set as the take profit target.
    stop_loss_pct (float): Percentage of the range to set as the stop loss level.

    Returns:
    pandas.DataFrame: A dataframe containing the details of each trade executed.
    """
    # Ensure the dataframe is sorted by index
    df = df.sort_index()

    # Create a unique ID for each Asia open session
    df['Asia_Open_Session_ID'] = (df['Asia_Open_Zone'] & (~df['Asia_Open_Zone'].shift(1).fillna(False))).cumsum()

    # Compute the high and low for each Asia open session
    asia_open_ranges = df[df['Asia_Open_Zone']].groupby('Asia_Open_Session_ID').agg({
        'High': 'max',
        'Low': 'min'
    }).rename(columns={'High': 'Asia_Open_High', 'Low': 'Asia_Open_Low'})

    # Forward-fill the session IDs and merge the Asia open ranges back into the dataframe
    df['Asia_Open_Session_ID'] = df['Asia_Open_Session_ID'].ffill()
    df = df.merge(asia_open_ranges, on='Asia_Open_Session_ID', how='left')

    # Calculate the range and breakout thresholds
    df['Asia_Open_Range'] = df['Asia_Open_High'] - df['Asia_Open_Low']
    df['Breakout_Threshold'] = breakout_pct * df['Asia_Open_Range']

    # Determine if each timestamp is after the Asia open session
    df['After_Asia_Open_Session'] = ~df['Asia_Open_Zone']

    # Initialize variables
    all_trades = []
    capital = starting_capital

    # Process each Asia open session individually
    for session_id, group in df.groupby('Asia_Open_Session_ID'):
        if capital <= 0:
            print("Capital exhausted. Stopping simulation.")
            break

        # Check if capital is sufficient for the margin
        num_contracts = capital // margin_per_contract
        if num_contracts == 0:
            print(f"Insufficient capital for margin in session {session_id}. Skipping trade.")
            continue

        trades = process_trading_strategy_futures(
            group,
            breakout_pct,
            capital,
            margin_per_contract,
            contract_size,
            take_profit_pct,
            stop_loss_pct,
            enter_opposite_side,
        )

        for trade in trades:
            # Update capital based on the profit from the trade
            capital += trade['Profit']
            # Store capital after the trade
            trade['Capital_After_Trade'] = capital

        all_trades.extend(trades)

    # Convert the list of trades into a DataFrame
    trades_df = pd.DataFrame(all_trades)
    return trades_df, df

def process_trading_strategy_futures(
    group,
    breakout_pct,
    capital,
    margin_per_contract,
    contract_size,
    take_profit_pct,
    stop_loss_pct,
    enter_opposite_side=False,
):
    """
    Process the trading strategy for a single Asia open session with futures contract considerations.

    Args:
    group (pandas.DataFrame): The dataframe for a single Asia open session.
    breakout_pct (float): Percentage of the range to consider as a breakout.
    capital (float): The capital available before the trade.
    margin_per_contract (float): The margin required per contract.
    contract_size (int): The contract size (e.g., 10 troy ounces for MGC).
    take_profit_pct (float): Percentage of the range to set as the take profit target.
    stop_loss_pct (float): Percentage of the range to set as the stop loss level.
    enter_opposite_side (bool): If True, enter at the opposite side of the range after first breakout.

    Returns:
    list: A list of dictionaries containing trade details.
    """
    # Only process data after the Asia open session
    group = group[group['After_Asia_Open_Session']]

    # Initialize variables
    in_breakout = False
    breakout_direction = None  # 'up' or 'down'
    reentered_range = False
    trade_executed = False
    trades = []

    # Get the Asia open high, low, and range
    if group.empty:
        return trades  # No data to process

    asia_open_high = group['Asia_Open_High'].iloc[0]
    asia_open_low = group['Asia_Open_Low'].iloc[0]
    asia_open_range = asia_open_high - asia_open_low
    breakout_threshold = breakout_pct * asia_open_range

    # Define breakout levels
    breakout_up_level = asia_open_high + breakout_threshold
    breakout_down_level = asia_open_low - breakout_threshold

    # Before entering a trade, check if capital is sufficient
    num_contracts = capital // margin_per_contract
    if num_contracts == 0:
        print(f"Insufficient capital for margin in session {group['Asia_Open_Session_ID'].iloc[0]}.")
        return trades  # No trade executed

    for idx, row in group.iterrows():
        if not in_breakout:
            # Check for initial breakout
            if row['High'] >= breakout_up_level:
                in_breakout = True
                breakout_direction = 'up'
                breakout_price = row['Close']
            elif row['Low'] <= breakout_down_level:
                in_breakout = True
                breakout_direction = 'down'
                breakout_price = row['Close']
        elif not reentered_range:
            # Check if price re-enters the range
            if (breakout_direction == 'up' and row['Low'] <= asia_open_low):
                reentered_range = True
            elif (breakout_direction == 'down' and row['High'] >= asia_open_high):
                reentered_range = True
        elif not trade_executed:
            if enter_opposite_side:
                # Enter at the opposite side of the range after re-entry
                if breakout_direction == 'up' and row['Low'] <= asia_open_low:
                    # Execute a long trade at the bottom of the range
                    trade_executed = True
                    entry_price = asia_open_low
                    trade_time = idx
                    stop_loss = entry_price - stop_loss_pct * asia_open_range
                    take_profit = entry_price + take_profit_pct * asia_open_range
                    trades.append({
                        'Time': trade_time,
                        'Direction': 'Long',
                        'Entry_Price': entry_price,
                        'Stop_Loss': stop_loss,
                        'Take_Profit': take_profit,
                        'Exit_Price': np.nan,
                        'Exit_Time': np.nan,
                        'Profit': np.nan,
                        'Capital_Before_Trade': capital,
                        'Margin_Per_Contract': margin_per_contract,
                        'Num_Contracts': num_contracts,
                        'Asia_Open_Session_ID': group['Asia_Open_Session_ID'].iloc[0]
                    })
                elif breakout_direction == 'down' and row['High'] >= asia_open_high:
                    # Execute a short trade at the top of the range
                    trade_executed = True
                    entry_price = asia_open_high
                    trade_time = idx
                    stop_loss = entry_price + stop_loss_pct * asia_open_range
                    take_profit = entry_price - take_profit_pct * asia_open_range
                    trades.append({
                        'Time': trade_time,
                        'Direction': 'Short',
                        'Entry_Price': entry_price,
                        'Stop_Loss': stop_loss,
                        'Take_Profit': take_profit,
                        'Exit_Price': np.nan,
                        'Exit_Time': np.nan,
                        'Profit': np.nan,
                        'Capital_Before_Trade': capital,
                        'Margin_Per_Contract': margin_per_contract,
                        'Num_Contracts': num_contracts,
                        'Asia_Open_Session_ID': group['Asia_Open_Session_ID'].iloc[0]
                    })
            else:
                # Original logic: Check for second breakout in the same direction
                if breakout_direction == 'up' and row['Low'] <= asia_open_low:
                    # Execute a long trade
                    trade_executed = True
                    entry_price = asia_open_low
                    trade_time = idx
                    stop_loss = entry_price - stop_loss_pct * asia_open_range
                    take_profit = asia_open_high + take_profit_pct * asia_open_range
                    trades.append({
                        'Time': trade_time,
                        'Direction': 'Long',
                        'Entry_Price': entry_price,
                        'Stop_Loss': stop_loss,
                        'Take_Profit': take_profit,
                        'Exit_Price': np.nan,
                        'Exit_Time': np.nan,
                        'Profit': np.nan,
                        'Capital_Before_Trade': capital,
                        'Margin_Per_Contract': margin_per_contract,
                        'Num_Contracts': num_contracts,
                        'Asia_Open_Session_ID': group['Asia_Open_Session_ID'].iloc[0]
                    })
                elif breakout_direction == 'down' and row['High'] >= asia_open_high:
                    # Execute a short trade
                    trade_executed = True
                    entry_price = asia_open_high
                    trade_time = idx
                    stop_loss = entry_price + stop_loss_pct * asia_open_range
                    take_profit = asia_open_low - take_profit_pct * asia_open_range
                    trades.append({
                        'Time': trade_time,
                        'Direction': 'Short',
                        'Entry_Price': entry_price,
                        'Stop_Loss': stop_loss,
                        'Take_Profit': take_profit,
                        'Exit_Price': np.nan,
                        'Exit_Time': np.nan,
                        'Profit': np.nan,
                        'Capital_Before_Trade': capital,
                        'Margin_Per_Contract': margin_per_contract,
                        'Num_Contracts': num_contracts,
                        'Asia_Open_Session_ID': group['Asia_Open_Session_ID'].iloc[0]
                    })
        elif trade_executed:
            # Manage the trade by checking stop loss and take profit
            current_trade = trades[-1]
            if current_trade['Direction'] == 'Long':
                # Check if take profit is hit
                if row['High'] >= current_trade['Take_Profit']:
                    exit_price = current_trade['Take_Profit']
                    exit_time = idx
                    profit = num_contracts * (exit_price - current_trade['Entry_Price']) * contract_size
                    current_trade.update({
                        'Exit_Price': exit_price,
                        'Exit_Time': exit_time,
                        'Profit': profit
                    })
                    break  # Exit after trade is closed
                # Check if stop loss is hit
                elif row['Low'] <= current_trade['Stop_Loss']:
                    exit_price = current_trade['Stop_Loss']
                    exit_time = idx
                    profit = num_contracts * (exit_price - current_trade['Entry_Price']) * contract_size
                    current_trade.update({
                        'Exit_Price': exit_price,
                        'Exit_Time': exit_time,
                        'Profit': profit
                    })
                    break  # Exit after trade is closed
            elif current_trade['Direction'] == 'Short':
                # Check if take profit is hit
                if row['Low'] <= current_trade['Take_Profit']:
                    exit_price = current_trade['Take_Profit']
                    exit_time = idx
                    profit = num_contracts * (current_trade['Entry_Price'] - exit_price) * contract_size
                    current_trade.update({
                        'Exit_Price': exit_price,
                        'Exit_Time': exit_time,
                        'Profit': profit
                    })
                    break  # Exit after trade is closed
                # Check if stop loss is hit
                elif row['High'] >= current_trade['Stop_Loss']:
                    exit_price = current_trade['Stop_Loss']
                    exit_time = idx
                    profit = num_contracts * (current_trade['Entry_Price'] - exit_price) * contract_size
                    current_trade.update({
                        'Exit_Price': exit_price,
                        'Exit_Time': exit_time,
                        'Profit': profit
                    })
                    break  # Exit after trade is closed

    # If trade is still open at the end of the day, exit at the close price
    if trade_executed and np.isnan(trades[-1]['Exit_Price']):
        current_trade = trades[-1]
        exit_price = group['Close'].iloc[-1]
        exit_time = group.index[-1]
        if current_trade['Direction'] == 'Long':
            profit = num_contracts * (exit_price - current_trade['Entry_Price']) * contract_size
        elif current_trade['Direction'] == 'Short':
            profit = num_contracts * (current_trade['Entry_Price'] - exit_price) * contract_size
        current_trade.update({
            'Exit_Price': exit_price,
            'Exit_Time': exit_time,
            'Profit': profit
        })

    return trades


# In your main function or script
if __name__ == "__main__":
    # Simulate the trading strategy with futures contract considerations
    trades_df, df = simulate_trading_strategy_futures(
        all_data,
        breakout_pct=1,
        starting_capital=10000,
        margin_per_contract=2060.49,  # Adjust as per current margin requirements
        contract_size=10,
        take_profit_pct=2.5,
        stop_loss_pct=1,
        enter_opposite_side=True,
    )

    # Calculate total profit
    total_profit = trades_df['Profit'].sum()
    print(f"Total Profit: {total_profit}")

    # Calculate win rate
    win_rate = (trades_df['Profit'] > 0).mean() * 100
    print(f"Win Rate: {win_rate:.2f}%")

    # Display the trades
    print(trades_df)

    # Plot the trading chart
    plot_trading_chart(df, trades_df)


Capital exhausted. Stopping simulation.
Total Profit: -34020.3515625
Win Rate: 50.00%
   Time Direction   Entry_Price     Stop_Loss   Take_Profit    Exit_Price  \
0  1125     Short  59634.777344  60302.769531  57964.796875  59193.296875   
1  1650      Long  58924.578125  58527.042969  59918.416016  58527.042969   

   Exit_Time        Profit  Capital_Before_Trade  Margin_Per_Contract  \
0       1195  17659.218750           10000.00000              2060.49   
1       1651 -51679.570312           27659.21875              2060.49   

   Num_Contracts  Asia_Open_Session_ID  Capital_After_Trade  
0            4.0                     4         27659.218750  
1           13.0                     6        -24020.351562  
df:                Open          High           Low         Close    Volume  \
0      64184.835938  64184.835938  64124.210938  64124.210938         0   
1      64083.449219  64083.449219  63971.738281  63975.136719         0   
2      63977.746094  64073.472656  63977.746094